In [1]:
#### 01.04.2026 Comparing the day 2 and day 2 lapatinib data, stored in the 'analysis/manual_labelled_2' folder
####  labels in manual_label and 'pc_manual_label' slots in the D2. labels in the manual_label, and manual_label_2 slots 

#### Imports

In [2]:
import os
import pandas as pd
import numpy as np
from scipy.sparse import issparse
import scanpy as sc


from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

#### Load in the D2 dataset

In [3]:
os.chdir("/Users/stanleydale/user_generated/breault-lab/single-cell/analysis/manual_labelled_2")

d2_dz = sc.read_h5ad("d2_dz_manual_labels_2.h5ad")

In [4]:
d2_dz

AnnData object with n_obs × n_vars = 46210 × 27844
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'freemuxlet.identity', 'participant', 'Condition', 'Time_point', 'Treatment', 'ident', 'leiden', 'manual_labelled', 'manual_label', 'pc_manual_label'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'X_name', 'hvg', 'manual_label_colors', 'pc_manual_label_colors'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

#### Inspect the participants that form our replicates

In [5]:
d2_dz.obs['participant'].unique()

['Negative', 'H897', 'H896', 'H329', 'Doublet']
Categories (5, object): ['Doublet', 'H329', 'H896', 'H897', 'Negative']

In [6]:

## Remove the doublet and negative categories

removed_participant_labels = ["Doublet", "Negative"]
d2_dz = d2_dz[
    ~d2_dz.obs['participant'].isin(removed_participant_labels),
].copy()

#### Define the cell type we're analysing; PCs

In [7]:
cell_type = "PCs"
sub = d2_dz[d2_dz.obs["manual_label"] == cell_type].copy()
sub.obs["sample_id"] = sub.obs["participant"].astype(str)
sub.obs["pb_id"] = (
    sub.obs["sample_id"].astype(str)
    + "_"
    + sub.obs["Condition"].astype(str)
)
sub

AnnData object with n_obs × n_vars = 4755 × 27844
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'freemuxlet.identity', 'participant', 'Condition', 'Time_point', 'Treatment', 'ident', 'leiden', 'manual_labelled', 'manual_label', 'pc_manual_label', 'sample_id', 'pb_id'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'X_name', 'hvg', 'manual_label_colors'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

#### Create a raw counts object, and confirm it's shape (genes as columns and cells as rows)

In [8]:

counts = sub.layers["counts"]

if issparse(counts):
    counts = counts.tocsr()

## let's confirm the shape of the counts, expecting genes as columns and cells as rows
n_cells = sub.n_obs
n_genes = sub.n_vars
print(counts.shape)
print(n_cells, n_genes)

(4755, 27844)
4755 27844


#### Create a pd df from this sparse matrix

In [9]:
counts_df = pd.DataFrame.sparse.from_spmatrix(
    counts,
    index=sub.obs_names,
    columns=sub.var_names
)

counts_df

,MIR1302-2HG,AL627309.1,AL627309.3,AL627309.5,LINC01409,FAM87B,LINC01128,LINC00115,FAM41C,AL645608.6,...,AC011043.2,AC011841.1,AL354822.1,AL592183.1,AC240274.1,AC004556.3,AC136352.3,AC136616.1,AC007325.4,AC007325.2
AAACCAAAGCCAACTA-1,0,0,0,0,1.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCAAAGGCACACT-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCATTCAAGTGAA-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCCGCATCCCGAT-1,0,0,0,0,0,0,1.0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCCTGTATTGGGA-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1.0,0,0,0,0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TGTGTACGTTTCAGCG-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TGTGTTAGTCAACTTA-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TGTGTTAGTCATGCTG-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TGTGTTGAGGGCAGGT-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


#### Compute our pseudobulk values

In [10]:
pb_counts = counts_df.groupby(sub.obs["pb_id"]).sum()

#### Transpose the pandas df

In [11]:
pb_counts = pb_counts.T
pb_counts

pb_id,H329_G2D2,H896_G2D2,H897_G2D2
MIR1302-2HG,0,0,0
AL627309.1,3.0,0,9.0
AL627309.3,0,0,2.0
AL627309.5,1.0,2.0,15.0
LINC01409,4.0,4.0,184.0
...,...,...,...
AC004556.3,0,3.0,7.0
AC136352.3,0,0,0
AC136616.1,0,0,0
AC007325.4,2.0,0,31.0


#### Save to csv, we will retrieve it when we merge the pseudobulk df with the comparison pseudobulk df

In [12]:
pb_counts.to_csv("/Users/stanleydale/user_generated/breault-lab/single-cell/data/dge/pseudobulk-csvs/pb_counts_d2_pcs.csv")

#### Repeat for the D2 + Lapatinib data, before merging the pseudobulk dataframes

In [4]:
del d2_dz

In [5]:
d2_lapa = sc.read_h5ad("d2_lapa_manual_labels_2.h5ad")

In [6]:
## Remove the doublet and negative categories

removed_participant_labels = ["Doublet", "Negative"]
d2_lapa = d2_lapa[
    ~d2_lapa.obs['participant'].isin(removed_participant_labels),
].copy()

In [7]:
cell_type = "Secretory progenitor cells"
sub = d2_lapa[d2_lapa.obs["manual_label"] == cell_type].copy() ### Make sure to be explicit with the metadata slot
sub.obs["sample_id"] = sub.obs["participant"].astype(str)
sub.obs["pb_id"] = (
    sub.obs["sample_id"].astype(str)
    + "_"
    + sub.obs["Condition"].astype(str)
)
sub

AnnData object with n_obs × n_vars = 8320 × 28941
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'freemuxlet.identity', 'participant', 'Condition', 'Time_point', 'Treatment', 'ident', 'leiden', 'manual_labelled', 'manual_label', 'inflammed_manual_label', 'manual_label_2', 'sample_id', 'pb_id'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'X_name', 'hvg', 'leiden_colors', 'manual_label_2_colors'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [8]:

counts = sub.layers["counts"]

if issparse(counts):
    counts = counts.tocsr()

## let's confirm the shape of the counts, expecting genes as columns and cells as rows
n_cells = sub.n_obs
n_genes = sub.n_vars
print(counts.shape)
print(n_cells, n_genes)

(8320, 28941)
8320 28941


In [9]:
counts_df = pd.DataFrame.sparse.from_spmatrix(
    counts,
    index=sub.obs_names,
    columns=sub.var_names
)


In [10]:
pb_counts = counts_df.groupby(sub.obs["pb_id"]).sum()

In [11]:
pb_counts = pb_counts.T
pb_counts

pb_id,H329_G2D2_Lapa,H439_G2D2_Lapa,H896_G2D2_Lapa
AL627309.1,3.0,6.0,8.0
AL627309.5,15.0,19.0,25.0
AL627309.4,0,0,0
AP006222.2,0,0,0
LINC01409,113.0,160.0,147.0
...,...,...,...
AC240274.1,73.0,51.0,63.0
AC004556.3,0,1.0,96.0
AC136352.3,1.0,1.0,1.0
AC007325.4,24.0,40.0,25.0


#### Merge the pseudobulk dfs

In [12]:
original_pb = pd.read_csv("/Users/stanleydale/user_generated/breault-lab/single-cell/data/dge/pseudobulk-csvs/pb_counts_d2_pcs.csv", index_col=0)

In [13]:
original_pb

,H329_G2D2,H896_G2D2,H897_G2D2
MIR1302-2HG,0.0,0.0,0.0
AL627309.1,3.0,0.0,9.0
AL627309.3,0.0,0.0,2.0
AL627309.5,1.0,2.0,15.0
LINC01409,4.0,4.0,184.0
...,...,...,...
AC004556.3,0.0,3.0,7.0
AC136352.3,0.0,0.0,0.0
AC136616.1,0.0,0.0,0.0
AC007325.4,2.0,0.0,31.0


In [14]:
pb_counts

pb_id,H329_G2D2_Lapa,H439_G2D2_Lapa,H896_G2D2_Lapa
AL627309.1,3.0,6.0,8.0
AL627309.5,15.0,19.0,25.0
AL627309.4,0,0,0
AP006222.2,0,0,0
LINC01409,113.0,160.0,147.0
...,...,...,...
AC240274.1,73.0,51.0,63.0
AC004556.3,0,1.0,96.0
AC136352.3,1.0,1.0,1.0
AC007325.4,24.0,40.0,25.0


In [15]:
df_combined = pd.concat([original_pb, pb_counts], axis=1)

In [16]:
df_clean = df_combined.fillna(0)
df_clean = df_clean.astype(int)

In [17]:
counts_df = df_clean.T   # now: rows = samples, cols = genes
counts_df.shape

(6, 29512)

In [18]:
samples = counts_df.index

condition = ["Lapa" if "Lapa" in s else "Dz" for s in samples]
participant = [s.split("_")[0] for s in samples]

clinical_df = pd.DataFrame(
    {
        "condition": condition,
        "participant": participant,
    },
    index=samples,   # IMPORTANT: index must match counts_df.index
)

clinical_df

,condition,participant
H329_G2D2,Dz,H329
H896_G2D2,Dz,H896
H897_G2D2,Dz,H897
H329_G2D2_Lapa,Lapa,H329
H439_G2D2_Lapa,Lapa,H439
H896_G2D2_Lapa,Lapa,H896


#### Run pydeseq2

In [19]:
# create DESeq dataset
dds = DeseqDataSet(
    counts=counts_df,
    metadata=clinical_df,
    design_factors="condition",  # use the 'condition' column
)

# fit dispersions + LFCs
dds.deseq2()

# compare Lapa vs Dz
stats = DeseqStats(
    dds,
    contrast=["condition", "Lapa", "Dz"],  # tested_level, reference_level
)

stats.summary()
res = stats.results_df
res.head()

Using None as control genes, passed at DeseqDataSet initialization


/var/folders/gs/k55rgtm50g938k87mvcdclqr0000gq/T/ipykernel_1222/2214745823.py:2: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(
Fitting size factors...
... done in 0.01 seconds.

python(2639) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(2640) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(2641) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(2642) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(2643) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(2644) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(2645) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(2646) 

Log2 fold change & Wald test p-value: condition Lapa vs Dz
              baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
MIR1302-2HG   0.000000             NaN       NaN       NaN       NaN       NaN
AL627309.1    4.707037       -1.246597  1.148733 -1.085193  0.277836  0.566888
AL627309.3    0.096274       -3.646381  4.514326 -0.807736  0.419243       NaN
AL627309.5    8.373733        0.420167  0.818483  0.513348  0.607708  0.822255
LINC01409    47.873708        0.530762  0.447717  1.185485  0.235826  0.521595
...                ...             ...       ...       ...       ...       ...
FATE1         0.000000             NaN       NaN       NaN       NaN       NaN
ATP2B3        0.000000             NaN       NaN       NaN       NaN       NaN
HCFC1-AS1     0.000000             NaN       NaN       NaN       NaN       NaN
OPN1MW        0.000000             NaN       NaN       NaN       NaN       NaN
TEX28         0.061676       -2.199409  4.405382 -0.499255  0.617600    

... done in 0.71 seconds.



,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
MIR1302-2HG,0.000000,NaN,NaN,NaN,NaN,NaN
AL627309.1,4.707037,-1.246597,1.148733,-1.085193,0.277836,0.566888
AL627309.3,0.096274,-3.646381,4.514326,-0.807736,0.419243,NaN
AL627309.5,8.373733,0.420167,0.818483,0.513348,0.607708,0.822255
LINC01409,47.873708,0.530762,0.447717,1.185485,0.235826,0.521595


In [20]:
res.to_csv(
    "/Users/stanleydale/user_generated/breault-lab/single-cell/data/dge/pydeseq-output/d2_lapa_secretory_pcs_deseq2_results.csv",
    index=True
)